# Model 2 · Punctuation and casing cleanup

The whole Model 2 pipeline in one self-contained notebook: download WikiText-103 and
DialogSum, train the WordPiece tokenizer, build the noisy training arrays, train the
~7M-parameter transformer for every seed, evaluate on the test sets at every noise level
(with the rule baseline) and write the results table.

**Run**: Settings → Accelerator **GPU T4 x2**, Internet **on** → Run All.
Data + tokenizer take ~30 min, then training; the notebook stops cleanly before 11 h.

**Resume** after a cut-off session: *Add Input → Notebook Output* → this notebook's previous
version, then Run All. Data, checkpoints and results are copied back; finished steps and
seeds are skipped and an interrupted seed resumes from its last epoch.

**Output** (`/kaggle/working`):
- `checkpoints/model2/seed_<n>/best.keras` — best epoch on validation punctuation macro-F1
- `checkpoints/model2/tokenizer/` — the tokenizer the models were trained with
- `results/model2_cleanup/seed_<n>/` — `metrics.json`, `history.csv`, `manifest.json`
- `results/tables/model2_cleanup.md` — mean ± std over seeds
- `data/` — tokenizer and arrays (kept so a resumed session can skip rebuilding them)
- `manifest_model2_cleanup.json`

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"  # before any keras import

import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()
SMOKE = os.environ.get("SMOKE") == "1"  # tiny run for a quick CPU check (< 2 min)
WORK = Path(os.environ.get("WORK_DIR", "/kaggle/working"))
INPUT = Path(os.environ.get("INPUT_DIR", "/kaggle/input"))
STAGE = "model2_cleanup"
print(f"on_kaggle={ON_KAGGLE} smoke={SMOKE} work={WORK}")

In [ ]:
# Exact versions (Kaggle's preinstalled ones change over time).
import subprocess

if ON_KAGGLE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "jax[cuda12]==0.10.2", "keras==3.15.1",
         "datasets==5.1.0", "tokenizers==0.23.3"],
        check=True,
    )

## Configuration
Every hyperparameter of the run is here.

In [ ]:
CONFIG = {
    "name": "model2_cleanup",
    "seed": 0,                 # data seed (noise, arrays); shared by all training seeds
    "seeds": [0, 1, 2],        # training seeds
    "max_hours": 11,           # per Kaggle session (12 h cap), data build included
    "wikitext": {
        "hf_path": "Salesforce/wikitext",
        "hf_config": "wikitext-103-raw-v1",
        "revision": "b08601e04326c79dfdd32d625aee71d232d685c3",
    },
    "dialogsum": {
        "hf_path": "knkarthick/dialogsum",
        "revision": "a968e7aee0602e257935f1321a02e4287f7d5848",
        "max_speakers": 8,
    },
    "tokenizer": {
        "vocab_size": 16000,
        "min_frequency": 2,
        "limit_alphabet": 1000,
        "wordpieces_prefix": "##",
        "corpus": ["wikitext_train", "dialogsum_train"],
        "report_sources": ["wikitext_test", "dialogsum_test"],
        "max_items_per_source": None,
    },
    "data": {
        "window_words": 64,
        "max_len": 128,
        "train_sources": {"wikitext_train": 1, "dialogsum_train": 5},  # value = copies
        "validation_sources": ["wikitext_validation", "dialogsum_validation"],
        "test_sources": ["wikitext_test", "dialogsum_test"],
        "max_units": None,     # cap on articles / dialogues per source (smoke runs only)
        "noise": {"substitution": 0.05, "deletion": 0.03, "insertion": 0.02},
        "substitution_vocab_size": 20000,
    },
    "model": {
        "vocab_size": 16000,
        "d_model": 256,
        "num_layers": 4,
        "num_heads": 4,
        "ffn_dim": 1024,
        "max_len": 128,
        "dropout": 0.1,
        "init_std": 0.02,
    },
    "loss": {"punct_class_weighting": "inverse_sqrt_freq", "casing_weight": 1.0,
             "punct_weight": 1.0},
    "optimizer": {"peak_lr": 5.0e-4, "warmup_steps": 2000, "weight_decay": 0.01},
    "training": {
        "batch_size": 128,
        "epochs": 20,           # cosine decays to 0 over 8 epochs; early stopping may cut it
        "max_steps": None,
        "steps_per_epoch": None,  # None = one full pass over train
        "mixed_precision": "mixed_float16",
        "steps_per_execution": 32,  # throughput only: host syncs once per 32 steps
        "workers": 4,              # batch-prefetch threads
        "early_stopping": {"monitor": "val_punct_macro_f1", "mode": "max", "patience": 3},
    },
    "eval": {"batch_size": 256, "noise_levels": [0.0, 0.05, 0.10, 0.20]},
}
if SMOKE:
    CONFIG.update(name="model2_cleanup_smoke", seeds=[0], max_hours=0.03)
    CONFIG["tokenizer"]["max_items_per_source"] = 20000
    CONFIG["data"].update(max_units=60, window_words=16, max_len=32)
    CONFIG["model"].update(d_model=32, num_layers=1, num_heads=2, ffn_dim=64, max_len=32)
    CONFIG["optimizer"].update(peak_lr=3.0e-3, warmup_steps=5)
    CONFIG["training"].update(batch_size=8, epochs=5, max_steps=50, steps_per_epoch=10,
                              mixed_precision="float32", steps_per_execution=2, workers=2)
    CONFIG["eval"]["batch_size"] = 64

In [ ]:
import csv
import hashlib
import json
import math
import random
import re
import shutil
import time
import zlib
from collections import Counter, defaultdict
from dataclasses import dataclass, replace
from itertools import islice

import jax
import keras
import numpy as np
from keras import ops
from tokenizers import Tokenizer

SESSION_START = time.time()
print("python", sys.version.split()[0], "| keras", keras.__version__, "| jax", jax.__version__,
      "| backend", keras.backend.backend())
print("devices:", jax.devices())
if ON_KAGGLE and not SMOKE:
    assert jax.devices()[0].platform == "gpu", "No GPU: Settings -> Accelerator -> GPU T4 x2."

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
DATA = WORK / "data"
TOK_DIR = DATA / "tokenizer"
ARRAYS = DATA / "processed" / "model2"
CKPT = WORK / "checkpoints" / "model2"
RESULTS = WORK / "results"

## Resume
If a previous output of this notebook is attached, copy its data, checkpoints and results
back, whatever name Kaggle mounts it under.

In [ ]:
if INPUT.exists():
    for d, _, _ in os.walk(INPUT, followlinks=True):
        prev = Path(d)
        if (prev / f"manifest_{STAGE}.json").exists() or (prev / "checkpoints" / "model2").exists():
            for sub in ("data", "checkpoints", "results"):
                if (prev / sub).exists():
                    shutil.copytree(prev / sub, WORK / sub, dirs_exist_ok=True)
                    print("restored", sub, "from", prev)
NEED_DATA = not (ARRAYS / "meta.json").exists()
print("build data:", NEED_DATA)

## 1. DialogSum
Each `#PersonN#:` line is one utterance (tag removed). The test split stores each dialogue 3
times (one per reference); they are merged into `test_<n>`. Train dialogues whose text also
occurs in validation/test are dropped.

In [ ]:
SPEAKER_RE = re.compile(r"^\s*#person(\d+)#\s*:\s*", re.IGNORECASE)
TEST_ID_RE = re.compile(r"^(test_\d+)_(\d+)$")


def parse_dialogue(text, max_speakers):
    """Raw dialogue -> (speaker IDs, utterances). Untagged lines continue the previous turn."""
    turns = []
    for line in text.replace("\r\n", "\n").split("\n"):
        if not line.strip():
            continue
        match = SPEAKER_RE.match(line)
        if match is None:
            if not turns:
                raise ValueError(f"Text before the first speaker tag: {line[:60]!r}")
            turns[-1][1].append(line.strip())
            continue
        speaker = int(match.group(1))
        if not 1 <= speaker <= max_speakers:
            raise ValueError(f"Speaker {speaker} outside 1..{max_speakers}")
        first = line[match.end():].strip()
        turns.append((speaker, [first] if first else []))
    kept = [(s, " ".join(parts)) for s, parts in turns if parts]  # empty turns dropped
    if not kept:
        raise ValueError("Dialogue has no utterances.")
    return [s for s, _ in kept], [u for _, u in kept]


def group_test_rows(rows, max_speakers):
    """Merge rows test_<n>_<k> into one dialogue test_<n>."""
    groups = defaultdict(list)
    for row in rows:
        match = TEST_ID_RE.match(row["id"])
        if match is None:
            raise ValueError(f"Unexpected test ID: {row['id']!r}")
        groups[match.group(1)].append((int(match.group(2)), row))
    dialogues = []
    for dialogue_id in sorted(groups, key=lambda s: int(s.split("_")[1])):
        members = [row for _, row in sorted(groups[dialogue_id], key=lambda kr: kr[0])]
        if len({row["dialogue"].strip() for row in members}) != 1:
            raise ValueError(f"{dialogue_id}: rows have different dialogue text.")
        speakers, utterances = parse_dialogue(members[0]["dialogue"], max_speakers)
        dialogues.append({"id": dialogue_id, "speakers": speakers, "utterances": utterances})
    return dialogues


def load_dialogsum(cfg):
    from datasets import load_dataset

    ds = load_dataset(cfg["hf_path"], revision=cfg["revision"])
    out = {}
    for split in ("train", "validation", "test"):
        rows = list(ds[split])
        if split == "test":
            out[split] = group_test_rows(rows, cfg["max_speakers"])
            continue
        out[split] = []
        for r in rows:
            speakers, utterances = parse_dialogue(r["dialogue"], cfg["max_speakers"])
            out[split].append({"id": r["id"], "speakers": speakers, "utterances": utterances})
    ids = {s: {d["id"] for d in v} for s, v in out.items()}
    assert not (ids["train"] & ids["validation"] or ids["train"] & ids["test"]
                or ids["validation"] & ids["test"]), "Dialogue ID in more than one split"
    eval_texts = {"\n".join(d["utterances"]) for s in ("validation", "test") for d in out[s]}
    n = len(out["train"])
    out["train"] = [d for d in out["train"] if "\n".join(d["utterances"]) not in eval_texts]
    return out, n - len(out["train"])


if NEED_DATA:
    DIALOGSUM, DIALOGSUM_DROPPED = load_dialogsum(CONFIG["dialogsum"])
    print({s: len(v) for s, v in DIALOGSUM.items()}, "| train duplicates dropped:",
          DIALOGSUM_DROPPED)

## 2. WikiText-103 (raw)
Train articles whose title also appears in validation/test are dropped. The raw text is
still tokenised (`word , word .`, `@-@`); `detokenize` restores ordinary text.

In [ ]:
TITLE_RE = re.compile(r"^ = (?!=)(.*?) = $")
HEADING_RE = re.compile(r"^ (= )+.*( =)+ $")
AT_RE = re.compile(r" @([-,.])@ ")
ATTACH_LEFT = {",", ".", ";", ":", "!", "?", ")", "]", "}", "%"}
CLITIC_RE = re.compile(r"^'(s|t|m|d|re|ve|ll)$", re.IGNORECASE)
ATTACH_RIGHT = {"(", "[", "{", "$", "£", "€"}


def article_title(line):
    match = TITLE_RE.match(line.rstrip("\n"))
    return match.group(1) if match else None


def is_heading(line):
    return bool(HEADING_RE.match(line.rstrip("\n")))


def detokenize(line):
    """' a well @-@ known " test " , isn 't it ? ' -> 'a well-known "test", isn't it?'"""
    out, glue_next, quote_open = [], False, False
    for tok in AT_RE.sub(r"\1", f" {line.strip()} ").split():
        if tok == '"':
            attach = quote_open or glue_next
            quote_open = not quote_open
            glue_after = quote_open
        else:
            attach = glue_next or tok in ATTACH_LEFT or bool(CLITIC_RE.match(tok))
            glue_after = tok in ATTACH_RIGHT
        if out and attach:
            out[-1] += tok
        else:
            out.append(tok)
        glue_next = glue_after
    return " ".join(out)


def iter_articles(lines):
    """Each article as its detokenised paragraphs (headings and blank lines skipped)."""
    paragraphs = []
    for line in lines:
        if article_title(line) is not None:
            if paragraphs:
                yield paragraphs
            paragraphs = []
        elif line.strip() and not is_heading(line):
            paragraphs.append(detokenize(line))
    if paragraphs:
        yield paragraphs


def load_wikitext(cfg):
    from datasets import load_dataset

    ds = load_dataset(cfg["hf_path"], cfg["hf_config"], revision=cfg["revision"])
    wiki = {split: ds[split]["text"] for split in ("train", "validation", "test")}
    eval_titles = {t for s in ("validation", "test") for t in map(article_title, wiki[s]) if t}
    kept, dropping, n_dropped = [], False, 0
    for line in wiki["train"]:
        title = article_title(line)
        if title is not None:
            dropping = title in eval_titles
            n_dropped += dropping
        if not dropping:
            kept.append(line)
    wiki["train"] = kept
    return wiki, n_dropped


if NEED_DATA:
    WIKI, WIKI_DROPPED = load_wikitext(CONFIG["wikitext"])
    print({s: len(v) for s, v in WIKI.items()}, "lines | train articles dropped:", WIKI_DROPPED)

## 3. WordPiece tokenizer
Uncased BERT-style WordPiece, 16k vocab, trained on WikiText train + DialogSum train.
Special tokens have fixed IDs: `[PAD]=0 [UNK]=1 [CLS]=2 [SEP]=3 [MASK]=4 [S1]..[S8]=5..12`.
The alphabet is chosen deterministically and IDs are renumbered canonically (specials, then
sorted tokens), so the same corpus always gives the same `tokenizer.json`.

In [ ]:
PAD_ID, UNK_ID, CLS_ID, SEP_ID = 0, 1, 2, 3
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + [f"[S{i}]" for i in range(1, 9)]


def iter_source(source, max_items=None):
    """Texts of one source: WikiText paragraph lines or DialogSum utterances."""
    dataset, _, split = source.partition("_")
    if dataset == "wikitext":
        texts = (detokenize(x) for x in WIKI[split] if x.strip() and not is_heading(x))
    else:
        texts = (u for d in DIALOGSUM[split] for u in d["utterances"])
    return islice(texts, max_items)


def check_special_tokens(tokenizer):
    for expected, token in enumerate(SPECIAL_TOKENS):
        assert tokenizer.token_to_id(token) == expected, f"{token} has the wrong ID"


def train_tokenizer(cfg):
    from tokenizers import BertWordPieceTokenizer
    from tokenizers.models import WordPiece

    def corpus():
        for source in cfg["corpus"]:
            yield from iter_source(source, cfg["max_items_per_source"])

    wordpiece = BertWordPieceTokenizer(
        clean_text=True, handle_chinese_chars=True, strip_accents=True, lowercase=True,
        wordpieces_prefix=cfg["wordpieces_prefix"],
    )
    counts = Counter()
    for text in corpus():
        counts.update(wordpiece.normalizer.normalize_str(text))
    for space in [c for c in counts if c.isspace()]:
        del counts[space]
    ranked = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    alphabet = sorted(c for c, _ in ranked[: cfg["limit_alphabet"]])
    wordpiece.train_from_iterator(
        corpus(), vocab_size=cfg["vocab_size"], min_frequency=cfg["min_frequency"],
        limit_alphabet=len(alphabet), initial_alphabet=alphabet, special_tokens=SPECIAL_TOKENS,
        wordpieces_prefix=cfg["wordpieces_prefix"], show_progress=False,
    )
    tokenizer = wordpiece._tokenizer
    rest = sorted(set(tokenizer.get_vocab()) - set(SPECIAL_TOKENS))
    ordered = {tok: i for i, tok in enumerate([*SPECIAL_TOKENS, *rest])}
    tokenizer.model = WordPiece(ordered, unk_token="[UNK]",
                                continuing_subword_prefix=cfg["wordpieces_prefix"])
    check_special_tokens(tokenizer)
    return tokenizer


def tokenizer_report(tokenizer, texts):
    n_words = n_tokens = n_unk = 0
    texts = list(texts)
    for i in range(0, len(texts), 1000):
        for enc in tokenizer.encode_batch(texts[i:i + 1000], add_special_tokens=False):
            n_tokens += len(enc.ids)
            n_unk += sum(t == UNK_ID for t in enc.ids)
            n_words += len({w for w in enc.word_ids if w is not None})
    return {"texts": len(texts), "subwords_per_word": round(n_tokens / max(n_words, 1), 4),
            "unk_rate": round(n_unk / max(n_tokens, 1), 6)}


if NEED_DATA:
    cfg = CONFIG["tokenizer"]
    TOKENIZER = train_tokenizer(cfg)
    TOK_DIR.mkdir(parents=True, exist_ok=True)
    TOKENIZER.save(str(TOK_DIR / "tokenizer.json"))
    vocab = sorted(TOKENIZER.get_vocab().items(), key=lambda kv: kv[1])
    (TOK_DIR / "vocab.txt").write_text("".join(f"{t}\n" for t, _ in vocab), encoding="utf-8")
    report = {"vocab_size": TOKENIZER.get_vocab_size()}
    for source in cfg["report_sources"]:
        report[source] = tokenizer_report(TOKENIZER, iter_source(source, cfg["max_items_per_source"]))
    (TOK_DIR / "report.json").write_text(json.dumps(report, indent=2) + "\n")
    print(json.dumps(report, indent=2))
TOKENIZER = Tokenizer.from_file(str(TOK_DIR / "tokenizer.json"))
check_special_tokens(TOKENIZER)
TOKENIZER_SHA256 = hashlib.sha256((TOK_DIR / "tokenizer.json").read_bytes()).hexdigest()
assert TOKENIZER.get_vocab_size() <= CONFIG["model"]["vocab_size"]
print("tokenizer:", TOKENIZER.get_vocab_size(), "tokens, sha256", TOKENIZER_SHA256[:12])

## 4. Labels
Each whitespace token of the clean text gives one word: the model input `form` (lowercase,
punctuation removed; inner apostrophes/hyphens and digit separators kept), a `casing`
label (lower / capitalised / upper; `-1` = ignored for mixed case or no letters) and a
`punct` label for its trailing punctuation (none / comma / period / question;
`! ;` → period, `:` → comma). An abbreviation's final period belongs to the word.

In [ ]:
IGNORE = -1
CASING_CLASSES = ("lower", "capitalised", "upper")
PUNCT_CLASSES = ("none", "comma", "period", "question")
LOWER, CAPITALISED, UPPER = 0, 1, 2
NONE, COMMA, PERIOD, QUESTION = 0, 1, 2, 3
SENTENCE_END = {PERIOD, QUESTION}
PUNCT_MAP = {",": COMMA, ":": COMMA, ".": PERIOD, "!": PERIOD, ";": PERIOD, "?": QUESTION}
PRIORITY = (QUESTION, PERIOD, COMMA)
ABBREVIATIONS = frozenset(
    "mr mrs ms dr prof st jr sr vs etc inc ltd co corp no gen col lt sgt capt rev jan feb mar "
    "apr jun jul aug sep sept oct nov dec approx dept est fig vol".split()
)
EDGE_RE = re.compile(r"^(\W*)(.*?)(\W*)$", re.DOTALL)
INNER_RE = re.compile(r"(?<=[^\W_])['’\-](?=[^\W_])|(?<=\d)[.,](?=\d)")


@dataclass(frozen=True)
class Word:
    form: str
    casing: int
    punct: int


def casing_label(core):
    letters = [c for c in core if c.lower() != c.upper()]
    if not letters:
        return IGNORE
    if all(c.islower() for c in letters):
        return LOWER
    if len(letters) == 1 or letters[0].isupper() and all(c.islower() for c in letters[1:]):
        return CAPITALISED
    if all(c.isupper() for c in letters):
        return UPPER
    return IGNORE


def label_word(token):
    _, core, trailing = EDGE_RE.match(token).groups()
    is_abbrev = (core.lower() in ABBREVIATIONS or (any(c.isalpha() for c in core) and "." in core)
                 or (len(core) == 1 and core.isupper()))
    if trailing.startswith(".") and is_abbrev:
        core, trailing = core + ".", trailing[1:]
    kept = {m.start() for m in INNER_RE.finditer(core)}
    form = "".join(c for i, c in enumerate(core) if c.isalnum() or i in kept).lower()
    if not form:
        return None
    marks = {PUNCT_MAP[c] for c in trailing if c in PUNCT_MAP}
    return Word(form, casing_label(core), next((p for p in PRIORITY if p in marks), NONE))


def label_text(text):
    return [w for w in map(label_word, text.split()) if w is not None]


def unit_windows(text, window_words):
    """Label a unit, split it into sentences and pack whole sentences into windows."""
    sentences, current = [], []
    for word in label_text(text):
        current.append(word)
        if word.punct in SENTENCE_END:
            sentences.append(current)
            current = []
    if current:
        sentences.append(current)
    windows, current = [], []
    for sentence in sentences:
        if current and len(current) + len(sentence) > window_words:
            windows.append(current)
            current = []
        current = current + sentence
    if current:
        windows.append(current)
    return windows


for w in label_text('Mr. Smith said: "Hello, world!" Is it the U.S. or not?'):
    print(w)

## 5. Noise
Per word: substitution (random frequent word, labels kept), deletion (a punctuation label
moves to the previous word; a deleted sentence-initial capital moves to the next word),
insertion (random word before it, lower / none). Substitution and deletion are exclusive.

In [ ]:
@dataclass(frozen=True)
class Noise:
    substitution: float
    deletion: float
    insertion: float

    @property
    def total(self):
        return self.substitution + self.deletion + self.insertion

    def scaled(self, total):
        """Same ratios at another total level (test sets at several noise levels)."""
        if total == 0:
            return Noise(0.0, 0.0, 0.0)
        f = total / self.total
        return Noise(self.substitution * f, self.deletion * f, self.insertion * f)


def apply_noise(words, noise, vocab, rng):
    if noise.total == 0:
        return list(words)
    out, pending_capital, sentence_start = [], False, True
    draws = rng.random((len(words), 2))
    picks = rng.integers(0, len(vocab), size=(len(words), 2))
    for i, word in enumerate(words):
        starts_sentence = sentence_start
        sentence_start = word.punct in SENTENCE_END
        if draws[i, 1] < noise.insertion:
            out.append(Word(vocab[picks[i, 1]], LOWER, NONE))
        u = draws[i, 0]
        if u < noise.substitution:
            word = replace(word, form=vocab[picks[i, 0]])
        elif u < noise.substitution + noise.deletion:
            if word.punct != NONE and out:
                out[-1] = replace(out[-1], punct=word.punct)
            if starts_sentence and word.casing == CAPITALISED:
                pending_capital = True
            if word.punct in SENTENCE_END:
                pending_capital = False
            continue
        if pending_capital:
            if word.casing == LOWER:
                word = replace(word, casing=CAPITALISED)
            pending_capital = False
        out.append(word)
    return out

## 6. Arrays
Windows are tokenised as `[CLS] subwords [SEP]`, padded/truncated to `max_len`. Labels sit on
the first subword of each word, `-1` elsewhere. Set names: `train`, `validation_<dataset>`,
`test_<dataset>_noise<NN>`. Each set has its own random stream.

In [ ]:
def iter_units(source, max_units):
    """WikiText articles or DialogSum dialogues, as one text each."""
    dataset, _, split = source.partition("_")
    if dataset == "wikitext":
        units = (" ".join(p) for p in iter_articles(WIKI[split]))
    else:
        units = (" ".join(d["utterances"]) for d in DIALOGSUM[split])
    return islice(units, max_units)


def encode_windows(windows, max_len):
    n = len(windows)
    ids = np.full((n, max_len), PAD_ID, dtype=np.int32)
    casing = np.full((n, max_len), IGNORE, dtype=np.int8)
    punct = np.full((n, max_len), IGNORE, dtype=np.int8)
    truncated = total = 0
    if n == 0:
        return ids, casing, punct, 0, 0
    encodings = TOKENIZER.encode_batch([[w.form for w in win] for win in windows],
                                       is_pretokenized=True, add_special_tokens=False)
    budget = max_len - 2
    for row, (window, enc) in enumerate(zip(windows, encodings, strict=True)):
        toks = enc.ids[:budget]
        ids[row, 0] = CLS_ID
        ids[row, 1:1 + len(toks)] = toks
        ids[row, 1 + len(toks)] = SEP_ID
        seen = set()
        for pos, w in enumerate(enc.word_ids[:budget]):
            if w is not None and w not in seen:
                seen.add(w)
                casing[row, pos + 1] = window[w].casing
                punct[row, pos + 1] = window[w].punct
        total += len(window)
        truncated += len(window) - len(seen)
    return ids, casing, punct, truncated, total


def label_counts(labels, k):
    return np.bincount(labels[labels != IGNORE].astype(np.int64), minlength=k).tolist()


def build_set(name, sources, noise, vocab, chunk=20000):
    d_cfg = CONFIG["data"]
    rng = np.random.default_rng([CONFIG["seed"], zlib.crc32(name.encode())])
    parts, pending = [], []
    for source, copies in sources.items():
        for text in iter_units(source, d_cfg["max_units"]):
            windows = unit_windows(text, d_cfg["window_words"])
            for _ in range(copies):
                pending.extend(apply_noise(w, noise, vocab, rng) for w in windows)
            if len(pending) >= chunk:
                parts.append(encode_windows(pending, d_cfg["max_len"]))
                pending = []
    parts.append(encode_windows(pending, d_cfg["max_len"]))
    ids, casing, punct = (np.concatenate([p[i] for p in parts]) for i in range(3))
    d = ARRAYS / name
    d.mkdir(parents=True, exist_ok=True)
    np.save(d / "token_ids.npy", ids)
    np.save(d / "casing.npy", casing)
    np.save(d / "punct.npy", punct)
    meta = {
        "sources": sources, "noise": noise.__dict__, "noise_total": noise.total,
        "windows": int(ids.shape[0]), "max_len": int(ids.shape[1]),
        "words": sum(p[4] for p in parts), "truncated_words": sum(p[3] for p in parts),
        "casing_counts": label_counts(casing, 3), "punct_counts": label_counts(punct, 4),
    }
    (d / "meta.json").write_text(json.dumps(meta, indent=2) + "\n")
    print(f"{name}: {meta['windows']} windows, {meta['words']} words "
          f"({meta['truncated_words']} truncated)")
    return meta


def validation_set(source):
    return f"validation_{source.split('_')[0]}"


def test_set(source, level):
    return f"test_{source.split('_')[0]}_noise{round(level * 100):02d}"


if NEED_DATA:
    d_cfg = CONFIG["data"]
    ARRAYS.mkdir(parents=True, exist_ok=True)
    freq = Counter()
    for source in d_cfg["train_sources"]:  # each source counted once, not oversampled
        for text in iter_units(source, d_cfg["max_units"]):
            freq.update(w.form for w in label_text(text))
    ranked = sorted(freq.items(), key=lambda kv: (-kv[1], kv[0]))
    noise_vocab = [w for w, _ in ranked[: d_cfg["substitution_vocab_size"]]]
    (ARRAYS / "noise_vocab.txt").write_text("".join(f"{w}\n" for w in noise_vocab), "utf-8")
    noise = Noise(**d_cfg["noise"])

    train_meta = build_set("train", dict(d_cfg["train_sources"]), noise, noise_vocab)
    for source in d_cfg["validation_sources"]:
        build_set(validation_set(source), {source: 1}, noise, noise_vocab)
    for source in d_cfg["test_sources"]:
        for level in CONFIG["eval"]["noise_levels"]:
            build_set(test_set(source, level), {source: 1}, noise.scaled(level), noise_vocab)

    # Punctuation class weights ∝ 1/sqrt(frequency), averaging 1 over labelled train positions.
    f = np.asarray(train_meta["punct_counts"], dtype=np.float64)
    f /= f.sum()
    w = np.zeros_like(f)
    w[f > 0] = 1.0 / np.sqrt(f[f > 0])
    w /= (f * w).sum()
    summary = {
        "casing_classes": list(CASING_CLASSES),
        "punct_classes": list(PUNCT_CLASSES),
        "punct_class_weights": [round(float(x), 6) for x in w],
        "train_punct_counts": train_meta["punct_counts"],
        "train_casing_counts": train_meta["casing_counts"],
        "tokenizer_sha256": TOKENIZER_SHA256,
        "data_config": {k: CONFIG[k] for k in ("seed", "wikitext", "dialogsum", "tokenizer", "data")},
        "dialogsum_train_duplicates_dropped": DIALOGSUM_DROPPED,
        "wikitext_train_articles_dropped": WIKI_DROPPED,
    }
    (ARRAYS / "meta.json").write_text(json.dumps(summary, indent=2) + "\n")
    del WIKI, DIALOGSUM  # free memory before training

DATA_META = json.loads((ARRAYS / "meta.json").read_text())
assert DATA_META["tokenizer_sha256"] == TOKENIZER_SHA256, "arrays and tokenizer do not match"
print("punct class weights:", DATA_META["punct_class_weights"])
print(f"data ready after {(time.time() - SESSION_START) / 60:.1f} min")

## 7. Model
Pre-LayerNorm transformer encoder (token + learned position embeddings), then two
token-classification heads (casing, punctuation) in float32. The layers are registered under
`dialogue_models` so the saved `.keras` files load wherever these classes are defined.

In [ ]:
def _init(std):
    return keras.initializers.TruncatedNormal(stddev=std)


@keras.saving.register_keras_serializable(package="dialogue_models")
class TokenAndPositionEmbedding(keras.layers.Layer):
    """Sum of token, learned position and (optionally) speaker embeddings, then dropout."""

    def __init__(self, vocab_size, max_len, d_model, num_speakers=0, dropout=0.1,
                 init_std=0.02, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.d_model = d_model
        self.num_speakers = num_speakers
        self.dropout_rate = dropout
        self.init_std = init_std
        self.token_embedding = keras.layers.Embedding(
            vocab_size, d_model, embeddings_initializer=_init(init_std), name="token_embedding")
        self.speaker_embedding = (
            keras.layers.Embedding(num_speakers + 1, d_model,
                                   embeddings_initializer=_init(init_std), name="speaker_embedding")
            if num_speakers else None
        )
        self.dropout = keras.layers.Dropout(dropout)

    def build(self, input_shape):
        self.token_embedding.build(input_shape)
        if self.speaker_embedding is not None:
            self.speaker_embedding.build(input_shape)
        self.position_embedding = self.add_weight(
            name="position_embedding", shape=(self.max_len, self.d_model),
            initializer=_init(self.init_std))
        super().build(input_shape)

    def call(self, token_ids, speaker_ids=None, training=None):
        length = ops.shape(token_ids)[-1]
        x = self.token_embedding(token_ids)
        x = x + ops.cast(self.position_embedding[:length], x.dtype)[None, :, :]
        if self.speaker_embedding is not None:
            if speaker_ids is None:
                raise ValueError("speaker_ids is required when num_speakers > 0")
            x = x + self.speaker_embedding(speaker_ids)
        return self.dropout(x, training=training)

    def compute_output_shape(self, input_shape):
        return (*input_shape, self.d_model)

    def get_config(self):
        config = super().get_config()
        config.update(vocab_size=self.vocab_size, max_len=self.max_len, d_model=self.d_model,
                      num_speakers=self.num_speakers, dropout=self.dropout_rate,
                      init_std=self.init_std)
        return config


@keras.saving.register_keras_serializable(package="dialogue_models")
class TransformerEncoderBlock(keras.layers.Layer):
    """Pre-LN block: x + MHA(LN(x)), then x + FFN(LN(x)); FFN = Dense -> GELU -> Dense."""

    def __init__(self, d_model, num_heads, ffn_dim, dropout=0.1, init_std=0.02, **kwargs):
        super().__init__(**kwargs)
        if d_model % num_heads:
            raise ValueError(f"d_model={d_model} is not divisible by num_heads={num_heads}")
        self.d_model = d_model
        self.num_heads = num_heads
        self.ffn_dim = ffn_dim
        self.dropout_rate = dropout
        self.init_std = init_std
        self.attn_norm = keras.layers.LayerNormalization(epsilon=1e-6, name="attn_norm")
        self.attention = keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout,
            kernel_initializer=_init(init_std), name="attention")
        self.attn_dropout = keras.layers.Dropout(dropout)
        self.ffn_norm = keras.layers.LayerNormalization(epsilon=1e-6, name="ffn_norm")
        self.ffn_in = keras.layers.Dense(ffn_dim, activation="gelu",
                                         kernel_initializer=_init(init_std), name="ffn_in")
        self.ffn_out = keras.layers.Dense(d_model, kernel_initializer=_init(init_std),
                                          name="ffn_out")
        self.ffn_dropout = keras.layers.Dropout(dropout)

    def build(self, input_shape):
        self.attn_norm.build(input_shape)
        self.attention.build(input_shape, input_shape)
        self.ffn_norm.build(input_shape)
        self.ffn_in.build(input_shape)
        self.ffn_out.build((*input_shape[:-1], self.ffn_dim))
        super().build(input_shape)

    def call(self, x, padding_mask=None, training=None):
        mask = None
        if padding_mask is not None:  # queries attend to real keys only
            mask = ops.logical_and(padding_mask[:, :, None], padding_mask[:, None, :])
        h = self.attn_norm(x)
        h = self.attention(h, h, attention_mask=mask, training=training)
        x = x + self.attn_dropout(h, training=training)
        h = self.ffn_out(self.ffn_in(self.ffn_norm(x)))
        return x + self.ffn_dropout(h, training=training)

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        config = super().get_config()
        config.update(d_model=self.d_model, num_heads=self.num_heads, ffn_dim=self.ffn_dim,
                      dropout=self.dropout_rate, init_std=self.init_std)
        return config


@keras.saving.register_keras_serializable(package="dialogue_models")
class TransformerEncoder(keras.layers.Layer):
    """Embeddings, num_layers pre-LN blocks and a final LayerNorm. Token 0 is padding."""

    def __init__(self, vocab_size, max_len, d_model, num_layers, num_heads, ffn_dim,
                 dropout=0.1, init_std=0.02, num_speakers=0, pad_id=0, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.d_model = d_model
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.ffn_dim = ffn_dim
        self.dropout_rate = dropout
        self.init_std = init_std
        self.num_speakers = num_speakers
        self.pad_id = pad_id
        self.embedding = TokenAndPositionEmbedding(
            vocab_size, max_len, d_model, num_speakers, dropout, init_std, name="embedding")
        self.blocks = [
            TransformerEncoderBlock(d_model, num_heads, ffn_dim, dropout, init_std,
                                    name=f"block_{i}")
            for i in range(num_layers)
        ]
        self.final_norm = keras.layers.LayerNormalization(epsilon=1e-6, name="final_norm")

    def build(self, input_shape):
        self.embedding.build(input_shape)
        hidden = (*input_shape, self.d_model)
        for block in self.blocks:
            block.build(hidden)
        self.final_norm.build(hidden)
        super().build(input_shape)

    def call(self, token_ids, speaker_ids=None, training=None):
        padding_mask = ops.not_equal(token_ids, self.pad_id)
        x = self.embedding(token_ids, speaker_ids, training=training)
        for block in self.blocks:
            x = block(x, padding_mask=padding_mask, training=training)
        return self.final_norm(x)

    def compute_output_shape(self, input_shape):
        return (*input_shape, self.d_model)

    def get_config(self):
        config = super().get_config()
        config.update(vocab_size=self.vocab_size, max_len=self.max_len, d_model=self.d_model,
                      num_layers=self.num_layers, num_heads=self.num_heads,
                      ffn_dim=self.ffn_dim, dropout=self.dropout_rate, init_std=self.init_std,
                      num_speakers=self.num_speakers, pad_id=self.pad_id)
        return config


def build_cleanup_model(m):
    """token_ids -> {"casing": logits, "punct": logits}; heads in float32."""
    token_ids = keras.Input(shape=(None,), dtype="int32", name="token_ids")
    hidden = TransformerEncoder(
        vocab_size=m["vocab_size"], max_len=m["max_len"], d_model=m["d_model"],
        num_layers=m["num_layers"], num_heads=m["num_heads"], ffn_dim=m["ffn_dim"],
        dropout=m["dropout"], init_std=m["init_std"], name="encoder",
    )(token_ids)
    casing = keras.layers.Dense(len(CASING_CLASSES), kernel_initializer=_init(m["init_std"]),
                                dtype="float32", name="casing")(hidden)
    punct = keras.layers.Dense(len(PUNCT_CLASSES), kernel_initializer=_init(m["init_std"]),
                               dtype="float32", name="punct")(hidden)
    return keras.Model(token_ids, {"casing": casing, "punct": punct}, name="cleanup")


build_cleanup_model(CONFIG["model"]).summary()

## 8. Training utilities
Memory-mapped arrays, a shuffling batch generator with masked labels and per-position
weights, AdamW with warmup + cosine decay, and a time guard that stops cleanly before the
session limit.

In [ ]:
@dataclass
class ArraySet:
    token_ids: np.ndarray
    casing: np.ndarray
    punct: np.ndarray

    @classmethod
    def load(cls, d, mmap=True):
        mode = "r" if mmap else None
        return cls(*(np.load(d / f"{k}.npy", mmap_mode=mode)
                     for k in ("token_ids", "casing", "punct")))

    @classmethod
    def concat(cls, sets):
        return cls(*(np.concatenate([getattr(s, k) for s in sets])
                     for k in ("token_ids", "casing", "punct")))

    def __len__(self):
        return int(self.token_ids.shape[0])


def label_weights(labels, class_weights=None):
    """0 at ignored positions, else the class weight (or 1)."""
    valid = labels != IGNORE
    if class_weights is None:
        return valid.astype(np.float32)
    table = np.asarray(class_weights, dtype=np.float32)
    return np.where(valid, table[np.where(valid, labels, 0)], 0.0).astype(np.float32)


class CleanupDataset(keras.utils.PyDataset):
    """Batches (token_ids, labels, sample_weights), reshuffled every epoch."""

    def __init__(self, arrays, batch_size, punct_class_weights=None, shuffle=True, seed=0,
                 max_batches=None, drop_last=True, **kwargs):
        super().__init__(**kwargs)
        self.arrays, self.batch_size = arrays, batch_size
        self.weights, self.shuffle, self.seed, self.epoch = punct_class_weights, shuffle, seed, 0
        n = len(arrays)
        n_batches = n // batch_size if drop_last else math.ceil(n / batch_size)
        if n_batches == 0:
            raise ValueError(f"{n} windows is fewer than one batch of {batch_size}")
        self.n_batches = n_batches if max_batches is None else min(n_batches, max_batches)
        self._order = self._make_order()

    def _make_order(self):
        n = len(self.arrays)
        return np.random.default_rng([self.seed, self.epoch]).permutation(n) if self.shuffle \
            else np.arange(n)

    def __len__(self):
        return self.n_batches

    def __getitem__(self, index):
        idx = np.sort(self._order[index * self.batch_size:(index + 1) * self.batch_size])
        n_real = len(idx)
        if n_real < self.batch_size:  # always full batches: data parallel splits them evenly
            idx = np.concatenate([idx, np.repeat(idx[-1:], self.batch_size - n_real)])
        casing = np.asarray(self.arrays.casing[idx])
        punct = np.asarray(self.arrays.punct[idx])
        y = {"casing": np.where(casing == IGNORE, 0, casing).astype(np.int32),
             "punct": np.where(punct == IGNORE, 0, punct).astype(np.int32)}
        sw = {"casing": label_weights(casing), "punct": label_weights(punct, self.weights)}
        for w in sw.values():
            w[n_real:] = 0.0  # padding rows do not count
        return np.asarray(self.arrays.token_ids[idx], dtype=np.int32), y, sw

    def set_epoch(self, epoch):
        """Called by SyncEpoch: after a resume Keras restarts at the true epoch, not 0."""
        if epoch != self.epoch:
            self.epoch = epoch
            self._order = self._make_order()


class SyncEpoch(keras.callbacks.Callback):
    def __init__(self, dataset):
        super().__init__()
        self.dataset = dataset

    def on_epoch_begin(self, epoch, logs=None):
        self.dataset.set_epoch(epoch)


def masked_sparse_ce():
    return keras.losses.SparseCategoricalCrossentropy(
        from_logits=True, reduction="mean_with_sample_weight")


def build_optimizer(o, total_steps):
    """AdamW, linear warmup to peak_lr, cosine decay to 0 at total_steps."""
    warmup = min(int(o["warmup_steps"]), max(total_steps - 1, 0))
    schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.0, decay_steps=max(total_steps - warmup, 1),
        warmup_target=o["peak_lr"], warmup_steps=warmup)
    optimizer = keras.optimizers.AdamW(learning_rate=schedule, weight_decay=o["weight_decay"])
    optimizer.exclude_from_weight_decay(var_names=["bias", "gamma", "beta", "position_embedding"])
    return optimizer


class TimeGuard(keras.callbacks.Callback):
    """Stop before max_hours after SESSION_START (shared by all seeds of the session).

    The hard limit is checked on every batch-end call: with steps_per_execution > 1 Keras
    calls it once per execution, so `batch` is not a multiple of any fixed interval."""

    def __init__(self, max_hours):
        super().__init__()
        self.limit = max_hours * 3600.0
        self.stopped, self._epoch_start, self._longest = False, 0.0, 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_train_batch_end(self, batch, logs=None):
        if time.time() - SESSION_START > self.limit:
            print("TimeGuard: hard limit reached mid-epoch, stopping.")
            self.stopped = self.model.stop_training = True

    def on_epoch_end(self, epoch, logs=None):
        now = time.time()
        self._longest = max(self._longest, now - self._epoch_start)
        if now - SESSION_START + 1.2 * self._longest > self.limit:
            print("TimeGuard: the next epoch would overrun the session, stopping.")
            self.stopped = self.model.stop_training = True


def best_logged_value(csv_path, key, mode):
    """Best value of key so far in the CSV log (so a resumed checkpoint keeps the best)."""
    if not csv_path.exists():
        return None
    with csv_path.open() as f:
        values = [float(r[key]) for r in csv.DictReader(f) if r.get(key)]
    return (max(values) if mode == "max" else min(values)) if values else None

## 9. Evaluation
Word-level scores on labelled first-subword positions only. Punctuation macro-F1 is over
comma / period / question; casing macro-F1 over capitalised / upper. The rule baseline
capitalises the first word and "i", and puts a period after the last word.

In [ ]:
PUNCT_MACRO = ("comma", "period", "question")
CASING_MACRO = ("capitalised", "upper")


def classification_report(y_true, y_pred, names, macro_over):
    k = len(names)
    conf = np.bincount(np.asarray(y_true, np.int64) * k + np.asarray(y_pred, np.int64),
                       minlength=k * k).reshape(k, k)
    tp, predicted, support = np.diag(conf).astype(float), conf.sum(0), conf.sum(1)
    per_class = {}
    for i, name in enumerate(names):
        p = tp[i] / predicted[i] if predicted[i] else 0.0
        r = tp[i] / support[i] if support[i] else 0.0
        per_class[name] = {"precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0,
                           "support": int(support[i])}
    index = {n: i for i, n in enumerate(names)}
    seen = [n for n in macro_over if conf[index[n]].sum() or conf[:, index[n]].sum()]
    total = conf.sum()
    return {"per_class": per_class,
            "accuracy": float(np.trace(conf) / total) if total else 0.0,
            "macro_f1": float(np.mean([per_class[n]["f1"] for n in seen])) if seen else 0.0,
            "n": int(total)}


def predict(model, token_ids, batch_size):
    """Argmax per position. Inputs are padded to whole batches (data parallel splits every
    batch evenly across the GPUs) and the padding is dropped from the output."""
    ids = np.asarray(token_ids)
    n = len(ids)
    if n % batch_size:
        ids = np.concatenate([ids, np.repeat(ids[-1:], batch_size - n % batch_size, axis=0)])
    out = model.predict(ids, batch_size=batch_size, verbose=0)
    return {k: np.argmax(v[:n], axis=-1) for k, v in out.items()}


def score(arrays, pred):
    report = {}
    for head, names, macro in (("punct", PUNCT_CLASSES, PUNCT_MACRO),
                               ("casing", CASING_CLASSES, CASING_MACRO)):
        true = np.asarray(getattr(arrays, head))
        valid = true != IGNORE
        report[head] = classification_report(true[valid], pred[head][valid], names, macro)
    return report


def rule_baseline(arrays):
    ids = np.asarray(arrays.token_ids)
    words = np.asarray(arrays.punct) != IGNORE  # every kept word has a punct label
    is_continuation = np.zeros(TOKENIZER.get_vocab_size(), dtype=bool)
    for tok, idx in TOKENIZER.get_vocab().items():
        is_continuation[idx] = tok.startswith("##")
    next_ids = np.concatenate([ids[:, 1:], np.zeros_like(ids[:, :1])], axis=1)
    casing, punct = np.zeros_like(ids), np.zeros_like(ids)
    casing[words & (ids == TOKENIZER.token_to_id("i")) & ~is_continuation[next_ids]] = CAPITALISED
    rows = np.flatnonzero(words.any(axis=1))
    first = np.argmax(words[rows], axis=1)
    last = words.shape[1] - 1 - np.argmax(words[rows, ::-1], axis=1)
    casing[rows, first] = CAPITALISED
    punct[rows, last] = PERIOD
    return {"casing": casing, "punct": punct}


def evaluate_all(model):
    results = {}
    for source in CONFIG["data"]["test_sources"]:
        for level in CONFIG["eval"]["noise_levels"]:
            name = test_set(source, level)
            arrays = ArraySet.load(ARRAYS / name)
            entry = {"noise": level, "baseline": score(arrays, rule_baseline(arrays)),
                     "model": score(arrays, predict(model, arrays.token_ids,
                                                    CONFIG["eval"]["batch_size"]))}
            results[name] = entry
            print(f"{name}: model punct macro-F1 {entry['model']['punct']['macro_f1']:.4f} | "
                  f"baseline {entry['baseline']['punct']['macro_f1']:.4f}")
    return results


class ValidationMetrics(keras.callbacks.Callback):
    """Adds word-level val_punct_macro_f1 and casing scores to the epoch logs."""

    def __init__(self, arrays, batch_size):
        super().__init__()
        self.arrays, self.batch_size = arrays, batch_size

    def on_epoch_end(self, epoch, logs=None):
        report = score(self.arrays, predict(self.model, self.arrays.token_ids, self.batch_size))
        logs = logs if logs is not None else {}
        logs["val_punct_macro_f1"] = report["punct"]["macro_f1"]
        logs["val_casing_accuracy"] = report["casing"]["accuracy"]
        logs["val_casing_macro_f1"] = report["casing"]["macro_f1"]

## 10. Train every seed
A seed with `metrics.json` is complete and skipped. A seed cut by the time guard keeps its
`backup/` and resumes from its last epoch in the next session.

In [ ]:
def json_dump(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, default=str) + "\n")


def train_seed(seed):
    out, res = CKPT / f"seed_{seed}", RESULTS / CONFIG["name"] / f"seed_{seed}"
    tr, es = CONFIG["training"], CONFIG["training"]["early_stopping"]
    start = time.time()
    random.seed(seed)
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)
    keras.mixed_precision.set_dtype_policy(tr["mixed_precision"])
    n_dev = len(jax.devices())
    for bs in (tr["batch_size"], CONFIG["eval"]["batch_size"]):
        assert bs % n_dev == 0, f"batch size {bs} must be divisible by the {n_dev} devices"
    if n_dev > 1:
        keras.distribution.set_distribution(keras.distribution.DataParallel())

    weights = (DATA_META["punct_class_weights"]
               if CONFIG["loss"]["punct_class_weighting"] == "inverse_sqrt_freq" else None)
    train = ArraySet.load(ARRAYS / "train")
    val = ArraySet.concat([ArraySet.load(ARRAYS / validation_set(s), mmap=False)
                           for s in CONFIG["data"]["validation_sources"]])
    spe = len(train) // tr["batch_size"]
    if tr["steps_per_epoch"]:
        spe = min(spe, tr["steps_per_epoch"])
    epochs = tr["epochs"]
    if tr["max_steps"]:
        spe = min(spe, tr["max_steps"])
        epochs = min(epochs, math.ceil(tr["max_steps"] / spe))
    train_ds = CleanupDataset(train, tr["batch_size"], weights, seed=seed, max_batches=spe,
                              workers=tr["workers"])
    val_ds = CleanupDataset(val, tr["batch_size"], weights, shuffle=False, drop_last=False)

    model = build_cleanup_model(CONFIG["model"])
    model.compile(
        optimizer=build_optimizer(CONFIG["optimizer"], spe * epochs),
        loss={"casing": masked_sparse_ce(), "punct": masked_sparse_ce()},
        loss_weights={"casing": CONFIG["loss"]["casing_weight"],
                      "punct": CONFIG["loss"]["punct_weight"]},
        steps_per_execution=tr["steps_per_execution"],
    )
    print(f"seed {seed}: {model.count_params():,} params, {spe} steps/epoch x {epochs} epochs")
    out.mkdir(parents=True, exist_ok=True)
    res.mkdir(parents=True, exist_ok=True)
    guard = TimeGuard(CONFIG["max_hours"])
    callbacks = [
        SyncEpoch(train_ds),
        ValidationMetrics(val, CONFIG["eval"]["batch_size"]),  # before the callbacks reading it
        guard,
        keras.callbacks.ModelCheckpoint(
            str(out / "best.keras"), monitor=es["monitor"], mode=es["mode"], save_best_only=True,
            initial_value_threshold=best_logged_value(res / "history.csv", es["monitor"],
                                                      es["mode"])),
        keras.callbacks.EarlyStopping(monitor=es["monitor"], mode=es["mode"],
                                      patience=es["patience"]),
        # Keras deletes the backup on any train end (time-guard stop included): keep it, and
        # remove it below only once the seed is complete.
        keras.callbacks.BackupAndRestore(str(out / "backup"), delete_checkpoint=False),
        keras.callbacks.CSVLogger(str(res / "history.csv"), append=True),
    ]
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks,
                        verbose=2)
    manifest = {
        "name": CONFIG["name"], "seed": seed, "config": CONFIG, "params": model.count_params(),
        "tokenizer_sha256": TOKENIZER_SHA256, "device": [str(d) for d in jax.devices()],
        "keras": keras.__version__, "jax": jax.__version__, "steps_per_epoch": spe,
        "epochs_this_session": len(history.history.get("loss", [])),
        "seconds": round(time.time() - start, 1),
    }
    if guard.stopped:
        manifest["status"] = "interrupted"
        json_dump(res / "manifest.json", manifest)
        print(f"seed {seed} interrupted by the time guard: rerun with this output attached.")
        return False
    shutil.rmtree(out / "backup", ignore_errors=True)
    best = keras.saving.load_model(out / "best.keras")
    val_report = score(val, predict(best, val.token_ids, CONFIG["eval"]["batch_size"]))
    json_dump(res / "metrics.json", {"seed": seed, "params": model.count_params(),
                                     "validation": val_report, "test": evaluate_all(best)})
    manifest.update(status="complete", seconds=round(time.time() - start, 1))
    json_dump(res / "manifest.json", manifest)
    print(f"seed {seed} done: val punct macro-F1 {val_report['punct']['macro_f1']:.4f}")
    return True


for seed in CONFIG["seeds"]:
    if (RESULTS / CONFIG["name"] / f"seed_{seed}" / "metrics.json").exists():
        print(f"seed {seed} already complete, skipping")
        continue
    if not train_seed(seed):
        break  # out of time for this session

## 11. Results table and manifest

In [ ]:
COLUMNS = [
    ("Punct macro-F1", ("punct", "macro_f1")),
    ("Comma F1", ("punct", "per_class", "comma", "f1")),
    ("Period F1", ("punct", "per_class", "period", "f1")),
    ("Question F1", ("punct", "per_class", "question", "f1")),
    ("Casing acc", ("casing", "accuracy")),
    ("Capitalised F1", ("casing", "per_class", "capitalised", "f1")),
    ("UPPER F1", ("casing", "per_class", "upper", "f1")),
]


def get(d, path):
    for key in path:
        d = d[key]
    return float(d)


def mean_std(values):
    if len(values) == 1:
        return f"{values[0]:.3f}"
    return f"{np.mean(values):.3f} ± {np.std(values, ddof=1):.3f}"


exp = CONFIG["name"]
seeds = {s: json.loads(p.read_text()) for s in CONFIG["seeds"]
         if (p := RESULTS / exp / f"seed_{s}" / "metrics.json").exists()}
print("complete seeds:", sorted(seeds), "of", CONFIG["seeds"])
if seeds:
    header = ["Test set", "Noise", "System", *[c for c, _ in COLUMNS]]
    lines = [f"# {exp}", "",
             f"Seeds: {sorted(seeds)} (mean ± std over seeds; the rule baseline is deterministic).",
             "", "| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    first = next(iter(seeds.values()))
    for set_name, entry in first["test"].items():
        dataset, noise_pct = set_name.split("_")[1], f"{entry['noise']:.0%}"
        base = [f"{get(entry['baseline'], p):.3f}" for _, p in COLUMNS]
        model = [mean_std([get(m["test"][set_name]["model"], p) for m in seeds.values()])
                 for _, p in COLUMNS]
        lines.append(f"| {dataset} | {noise_pct} | Rule baseline | " + " | ".join(base) + " |")
        lines.append(f"| {dataset} | {noise_pct} | Model ({exp}) | " + " | ".join(model) + " |")
    val = mean_std([m["validation"]["punct"]["macro_f1"] for m in seeds.values()])
    lines += ["", f"Validation punct macro-F1: {val}. Parameters: {first['params']:,}.", ""]
    table = RESULTS / "tables" / f"{exp}.md"
    table.parent.mkdir(parents=True, exist_ok=True)
    table.write_text("\n".join(lines))
    print(table.read_text())
if len(seeds) < len(CONFIG["seeds"]):
    print("Not finished: run again with this notebook's output attached to resume.")

# The tokenizer travels with the models.
shutil.copytree(TOK_DIR, CKPT / "tokenizer", dirs_exist_ok=True)
manifest = {
    "stage": STAGE, "smoke": SMOKE, "config": CONFIG, "complete_seeds": sorted(seeds),
    "tokenizer_sha256": TOKENIZER_SHA256, "device": [str(d) for d in jax.devices()],
    "session_hours": round((time.time() - SESSION_START) / 3600, 2),
    "finished": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
json_dump(WORK / f"manifest_{STAGE}.json", manifest)
print(json.dumps({k: v for k, v in manifest.items() if k != "config"}, indent=2))